In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
csv_path = Path.cwd() / "heart.csv"

if not csv_path.exists():
    csv_path = Path("Python Guidance/day31-45/heart.csv")

df = pd.read_csv(csv_path)
df.head()

In [ ]:
target = "HeartDisease"
failure_mode_columns = ["TWF", "HDF", "PWF", "OSF", "RNF"]

numeric_features = [
    "Age",
    "RestingBP",
    "Cholesterol",
    "MaxHR",
    "Oldpeak",
]
categorical_features = ["Sex", "ChestPainType", "RestingECG"]

X = df[numeric_features + categorical_features]
y = df[target]

In [ ]:
print("数据形状:", df.shape)
print("缺失值数量:", int(df.isna().sum().sum()))
print("\n目标变量分布:")
print(y.value_counts().rename(index={0: "正常", 1: "心脏病"}))
print(f"\n故障比例: {y.mean():.2%}")

In [ ]:
df[numeric_features].describe().T

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("训练集:", X_train.shape)
print("测试集:", X_test.shape)
print("训练集故障比例:", f"{y_train.mean():.2%}")
print("测试集故障比例:", f"{y_test.mean():.2%}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

## KNN

In [ ]:
knn = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=5, weights="uniform")),
    ]
)

knn.fit(X_train, y_train)

In [ ]:
y_pred = knn.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["正常", "心脏病"], digits=3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["正常", "心脏病"],
    cmap="Blues",
    values_format="d",
)
plt.title("KNN 混淆矩阵（K=5）")
plt.show()

In [ ]:
k_values = [1, 3, 5, 7, 9, 15, 21, 31]
rows = []

for k in k_values:
    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", KNeighborsClassifier(n_neighbors=k, weights="uniform")),
        ]
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rows.append(
        {
            "k": k,
            "accuracy": model.score(X_test, y_test),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred, zero_division=0),
        }
    )

k_results = pd.DataFrame(rows)
k_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_results["k"], k_results["precision"], marker="o", label="Precision")
plt.plot(k_results["k"], k_results["recall"], marker="o", label="Recall")
plt.plot(k_results["k"], k_results["f1"], marker="o", label="F1")
plt.xlabel("K")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("不同 K 值下的测试集表现")
plt.show()

In [ ]:
best_k = int(k_results.sort_values("f1", ascending=False).iloc[0]["k"])
best_f1 = k_results["f1"].max()

print(f"按 F1 选择的 K: {best_k}")
print(f"对应 F1: {best_f1:.3f}")

## SVM

In [ ]:
from sklearn.svm import SVC
import numpy as np
from sklearn.datasets import make_blobs

In [ ]:
X_demo, y_demo = make_blobs(
    n_samples=100,
    centers=[(2.0, 2.0), (-2.0, 6.0)],
    cluster_std=0.9,
    random_state=33,
)

demo_svm = SVC(kernel="linear", C=1.0)
demo_svm.fit(X_demo, y_demo)

x_min, x_max = X_demo[:, 0].min() - 1, X_demo[:, 0].max() + 1
y_min, y_max = X_demo[:, 1].min() - 1, X_demo[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 400),
    np.linspace(y_min, y_max, 400),
)

Z = demo_svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

In [ ]:
plt.figure(figsize=(7, 6))

plt.contour(
    xx,
    yy,
    Z,
    levels=[-1, 0, 1],
    colors=["gray", "black", "gray"],
    linestyles=["--", "-", "--"],
)

plt.scatter(
    X_demo[:, 0],
    X_demo[:, 1],
    c=y_demo,
    cmap="bwr",
    edgecolors="black",
    s=55,
)

plt.scatter(
    demo_svm.support_vectors_[:, 0],
    demo_svm.support_vectors_[:, 1],
    s=160,
    facecolors="none",
    edgecolors="gold",
    linewidths=2,
    label="Support vectors",
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Linear SVM: decision boundary and margin")
plt.legend()
plt.show()

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [ ]:
svm = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            SVC(
                kernel="linear",
                C=1.0,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

svm.fit(X_train, y_train)

In [ ]:
y_pred = svm.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["正常", "心脏病"], digits=3))

In [ ]:
rows = []

for class_weight in [None, "balanced"]:
    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                SVC(
                    kernel="linear",
                    C=1.0,
                    class_weight=class_weight,
                    random_state=42,
                ),
            ),
        ]
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rows.append(
        {
            "class_weight": str(class_weight),
            "accuracy": model.score(X_test, y_test),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred, zero_division=0),
        }
    )

class_weight_results = pd.DataFrame(rows)
class_weight_results

## xgboost_lightgbm

In [ ]:
from sklearn.compose import ColumnTransformer
import xgboost as xgb
import lightgbm as lgb

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val,
)

print("训练集:", X_train.shape, f"故障比例 {y_train.mean():.2%}")
print("验证集:", X_val.shape, f"故障比例 {y_val.mean():.2%}")
print("测试集:", X_test.shape, f"故障比例 {y_test.mean():.2%}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print("编码后训练集:", X_train_enc.shape)
print("编码后验证集:", X_val_enc.shape)
print("编码后测试集:", X_test_enc.shape)
print("特征名:", list(feature_names))

In [ ]:
def metrics_at_threshold(y_true, probability, threshold=0.5):
    y_pred = (probability >= threshold).astype(int)
    y_true_array = y_true.to_numpy()

    tp = int(((y_pred == 1) & (y_true_array == 1)).sum())
    tn = int(((y_pred == 0) & (y_true_array == 0)).sum())
    fp = int(((y_pred == 1) & (y_true_array == 0)).sum())
    fn = int(((y_pred == 0) & (y_true_array == 1)).sum())

    return {
        "threshold": threshold,
        "accuracy": float((y_pred == y_true_array).mean()),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "false_positives": fp,
        "false_negatives": fn,
    }


def evaluate_model(name, model, X_eval, y_eval, threshold=0.5):
    probability = model.predict_proba(X_eval)[:, 1]
    row = metrics_at_threshold(y_eval, probability, threshold)
    row["model"] = name
    return row


def print_model_report(name, model):
    y_pred = model.predict(X_test_enc)
    print(name)
    print(classification_report(y_test, y_pred, target_names=["正常", "心脏病"], digits=3))

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
)

xgb_model.fit(
    X_train_enc,
    y_train,
    eval_set=[(X_val_enc, y_val)],
    verbose=False,
)

print("XGBoost 早停后的最佳轮数:", xgb_model.best_iteration)

In [ ]:
print_model_report("XGBoost 测试集结果", xgb_model)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    xgb_model.predict(X_test_enc),
    display_labels=["正常", "故障"],
    cmap="Blues",
    values_format="d",
)
plt.title("XGBoost 混淆矩阵")
plt.show()

In [ ]:
lgbm_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary",
    importance_type="gain",
    random_state=42,
    n_jobs=-1,
    force_col_wise=True,
    verbosity=-1,
)

lgbm_model.fit(
    X_train_enc,
    y_train,
    eval_set=[(X_val_enc, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, verbose=False),
        lgb.log_evaluation(period=0),
    ],
)

print("LightGBM 早停后的最佳轮数:", lgbm_model.best_iteration_)

In [ ]:
print_model_report("LightGBM 测试集结果", lgbm_model)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    lgbm_model.predict(X_test_enc),
    display_labels=["正常", "故障"],
    cmap="Blues",
    values_format="d",
)
plt.title("LightGBM 混淆矩阵")
plt.show()

In [ ]:
baseline_results = pd.DataFrame(
    [
        evaluate_model("XGBoost", xgb_model, X_test_enc, y_test),
        evaluate_model("LightGBM", lgbm_model, X_test_enc, y_test),
    ]
)

baseline_results[[
    "model",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "false_positives",
    "false_negatives",
]]

In [ ]:
def plot_top_features(importances, title, ax):
    importance = pd.Series(importances, index=feature_names)
    importance = importance.sort_values(ascending=False).head(10)

    ax.barh(importance.index[::-1], importance.values[::-1])
    ax.set_title(title)
    ax.set_xlabel("Importance")
    ax.grid(True, axis="x", alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_top_features(xgb_model.feature_importances_, "XGBoost 特征重要度", axes[0])
plot_top_features(lgbm_model.feature_importances_, "LightGBM 特征重要度", axes[1])
plt.tight_layout()
plt.show()

## 逻辑回归

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
import seaborn as sns

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ]
)

In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=42,
                ),
            ),
        ]
    ),
    "KNN": Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                KNeighborsClassifier(n_neighbors=5, weights="distance"),
            ),
        ]
    ),
    "XGBoost": Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                xgb.XGBClassifier(
                    n_estimators=200,
                    learning_rate=0.1,
                    max_depth=3,
                    objective="binary:logistic",
                    eval_metric="auc",
                    tree_method="hist",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "LightGBM": Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                lgb.LGBMClassifier(
                    n_estimators=200,
                    learning_rate=0.1,
                    num_leaves=15,
                    min_child_samples=20,
                    objective="binary",
                    random_state=42,
                    n_jobs=-1,
                    force_col_wise=True,
                    verbosity=-1,
                ),
            ),
        ]
    ),
}

list(models.keys())

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

records = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    for fold_index, row in pd.DataFrame(scores).iterrows():
        record = {
            "model": model_name,
            "fold": fold_index + 1,
        }
        for metric_name in scoring:
            record[metric_name] = row[f"test_{metric_name}"]
        records.append(record)

cv_results = pd.DataFrame(records)
cv_results

In [ ]:
summary = (
    cv_results.groupby("model")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_precision=("precision", "mean"),
        std_precision=("precision", "std"),
        mean_recall=("recall", "mean"),
        std_recall=("recall", "std"),
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
    )
    .sort_values("mean_f1", ascending=False)
)

summary.round(4)

In [ ]:
best_model_name = summary.index[0]
best_mean_f1 = summary.loc[best_model_name, "mean_f1"]
best_std_f1 = summary.loc[best_model_name, "std_f1"]

print("按平均 F1 排序后的最优模型:", best_model_name)
print(f"平均 F1: {best_mean_f1:.4f}")
print(f"F1 标准差: {best_std_f1:.4f}")

In [ ]:
metrics_to_plot = ["f1", "recall", "precision", "pr_auc"]
metric_titles = {
    "f1": "F1",
    "recall": "Recall",
    "precision": "Precision",
    "pr_auc": "PR-AUC",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for metric, ax in zip(metrics_to_plot, axes.flat):
    sns.boxplot(
        data=cv_results,
        x="model",
        y=metric,
        ax=ax,
        hue="model",
        palette="Set2",
        legend=False,
    )
    sns.stripplot(
        data=cv_results,
        x="model",
        y=metric,
        ax=ax,
        color="black",
        alpha=0.55,
        size=5,
    )
    ax.set_title(metric_titles[metric])
    ax.set_xlabel("")
    ax.set_ylabel("Score")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
stability_view = summary.reset_index()

plt.figure(figsize=(8, 5))
for _, row in stability_view.iterrows():
    plt.errorbar(
        row["std_f1"],
        row["mean_f1"],
        fmt="o",
        markersize=8,
        capsize=4,
        label=row["model"],
    )
    plt.annotate(
        row["model"],
        (row["std_f1"], row["mean_f1"]),
        textcoords="offset points",
        xytext=(8, 6),
    )

plt.xlabel("F1 标准差（越小越稳定）")
plt.ylabel("平均 F1（越大越好）")
plt.title("模型平均表现与稳定性")
plt.grid(True, alpha=0.3)
plt.show()

## 超参数

In [ ]:
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)

### 网格搜索

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ]
)

base_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            lgb.LGBMClassifier(
                n_estimators=200,
                objective="binary",
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
                force_col_wise=True,
                verbosity=-1,
            ),
        ),
    ]
)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

In [ ]:
param_grid = {
    "model__num_leaves": [7, 15, 31],
    "model__learning_rate": [0.05, 0.1],
    "model__min_child_samples": [10, 20],
}

grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    verbose=0,
)

grid_search.fit(X_train, y_train)

In [ ]:
print("Grid Search 最佳参数:")
print(grid_search.best_params_)
print(f"\nGrid Search 最佳交叉验证 F1: {grid_search.best_score_:.4f}")

### 随机搜索

In [ ]:
param_distributions = {
    "model__num_leaves": [7, 15, 31, 63],
    "model__learning_rate": [0.03, 0.05, 0.1, 0.2],
    "model__min_child_samples": [5, 10, 20, 40],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

random_search = RandomizedSearchCV(
    estimator=base_pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    verbose=0,
)

random_search.fit(X_train, y_train)

In [ ]:
print("Random Search 最佳参数:")
print(random_search.best_params_)
print(f"\nRandom Search 最佳交叉验证 F1: {random_search.best_score_:.4f}")